# Gradio_Car_Detection

# Version 1. 차량 인식 > 차량 bbox > 이미지 내 차량 비율

In [ ]:
# 1️⃣ 필수 패키지 및 코랩 전용 설정
!pip install -q gradio ultralytics
import matplotlib
matplotlib.use('Agg')

from google.colab import drive
drive.mount('/content/drive')

import time
import torch
import torch.nn as nn
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import os
import io
from ultralytics import YOLO  # 🌟 오타가 수정된 부분입니다!
import gradio as gr

print("\n⚙️ 환경 준비 완료! AI 모델을 불러옵니다...")

# =========================================================
# 2️⃣ AI 모델 세팅
# =========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ['damaged_car', 'non_car', 'normal_car']

resnet_model = models.resnet50()
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(CLASS_NAMES))
resnet_path = "/content/drive/MyDrive/03. HDMF/(share)HDMF_AUTO_SPOKE/DATA/DJ_FINAL_DATASET/ResNet50/best_model.pth"

if os.path.exists(resnet_path):
    resnet_model.load_state_dict(torch.load(resnet_path, map_location=DEVICE))

resnet_model = resnet_model.to(DEVICE)
resnet_model.eval()

yolo_path = "/content/drive/MyDrive/03. HDMF/(pre_study)2026_HDMF_AUTO_SPOKE/SUBJECT/WEEK1_CAR_DETECTION/FINE_TUNING_MODEL/yolov8x_fine_tuning_5th/weights/best.pt"
yolo_model = YOLO(yolo_path) if os.path.exists(yolo_path) else None

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

# =========================================================
# 3️⃣ 분석 파이프라인 함수
# =========================================================
def process_vehicle_image(img):
    if img is None:
        return None, "이미지를 업로드해주세요."

    img = img.convert("RGB")
    img_w, img_h = img.size

    start_resnet = time.time()
    img_tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = resnet_model(img_tensor)
        _, pred_idx = torch.max(outputs, 1)
        pred_class = CLASS_NAMES[pred_idx.item()]
    if torch.cuda.is_available(): torch.cuda.synchronize()
    resnet_time = time.time() - start_resnet

    if "non_car" in pred_class.lower():
        report = f"🚫 [분석 중단] 차량이 아닙니다.\n\n- 판별 결과: {pred_class.upper()}\n- 서버 연산 속도: {resnet_time:.4f}초\n\n💡 리소스를 절약했습니다."
        return img, report

    start_yolo = time.time()
    results = yolo_model(img, verbose=False) if yolo_model else None
    if torch.cuda.is_available(): torch.cuda.synchronize()
    yolo_time = time.time() - start_yolo

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(img)

    if results is None or len(results[0].boxes) == 0:
        ax.axis('off')
        out_img = fig_to_pil(fig)
        plt.close(fig)
        return out_img, f"⚠️ 윤곽선을 찾지 못했습니다.\n- 판별 결과: {pred_class.upper()}"

    boxes = results[0].boxes
    largest_box = None
    max_area = 0
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        area = (x2 - x1) * (y2 - y1)
        if area > max_area:
            max_area = area
            largest_box = (x1, y1, x2, y2)

    x1, y1, x2, y2 = largest_box
    box_w, box_h = x2 - x1, y2 - y1
    ratio = ((box_w * box_h) / (img_w * img_h)) * 100

    rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=5, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1 - 15, f"Ratio: {ratio:.1f}%", color='black', backgroundcolor='lime', fontsize=18, fontweight='bold')
    ax.axis('off')

    out_img = fig_to_pil(fig)
    plt.close(fig)

    total_time = resnet_time + yolo_time
    report = f"✅ 차량 정밀 분석 완료!\n\n- 판별 상태: {pred_class.upper()}\n- 차량 면적 비율: {ratio:.1f}%\n- 총 연산 속도: {total_time:.4f}초\n  (ResNet 1차: {resnet_time:.3f}s + YOLO 2차: {yolo_time:.3f}s)"

    return out_img, report

# =========================================================
# 4️⃣ Gradio 웹 UI 실행 (크기 대폭 확대)
# =========================================================
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🚗 AI 기반 차량 파손 정밀 분석 시스템")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="📁 원본 이미지", height=500)
            analyze_btn = gr.Button("🔍 AI 분석 시작", variant="primary", size="lg")

        with gr.Column():
            output_image = gr.Image(type="pil", label="🤖 시각화 결과", height=500)
            output_text = gr.Textbox(label="📊 성능 리포트", lines=5)

    analyze_btn.click(fn=process_vehicle_image, inputs=input_image, outputs=[output_image, output_text])

print("\n🎉 실행 성공! 코랩 셀 높이가 커졌습니다.")
print("💡 모니터 전체 화면으로 보려면 아래의 'Running on public URL: https://어쩌구.gradio.live' 링크를 클릭하세요!")

app.launch(debug=True, inline=True, height=800, share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

⚙️ 환경 준비 완료! AI 모델을 불러옵니다...


/tmp/ipykernel_295/3107609897.py:123: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:



🎉 실행 성공! 코랩 셀 높이가 커졌습니다.
💡 모니터 전체 화면으로 보려면 아래의 'Running on public URL: https://어쩌구.gradio.live' 링크를 클릭하세요!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d4a21154721a135abd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
